# 📦 Optimisation des Achats — Réduction de la Valeur de Stock
## Modèle MILP | Python · Pyomo · CBC

---

### Contexte
La valeur totale du stock du site (MP + SF + PF) est d'environ **4,8 M€**.
L'objectif est de la ramener à **2,4 M€ (−50%)** d'ici fin septembre,
en agissant sur deux leviers :

| Levier | Variable | Description |
|--------|----------|-------------|
| **Gel des achats** | $O_{i,t} = 0$ | Ne pas passer de nouvelles commandes |
| **Actions de déstockage** | $DISP_{k,t} \geq 0$ | Retours fournisseurs, mises au rebut, transferts (liés aux provisions PFO/GIR) |

### Simplifications par rapport au modèle complet
- ❌ Pas de production (pas de BOM, pas de nomenclature, pas de MLS)
- ❌ Pas de contrainte de capacité de production
- ✅ Achats MP uniquement comme levier décisionnel
- ✅ Toutes références (MP + SF + PF) suivies dans les bilans de stock
- ✅ Demande client pour les PF (drainage naturel du stock)
- ✅ Pipeline SAP intégré (OC et WIP déjà engagés)

### Structure du notebook
| Section | Contenu |
|---------|---------|
| **1** | Imports et paramètres globaux |
| **2** | Modèle MILP Pyomo (ConcreteModel) |
| **3** | Chargement des données Excel |
| **4** | Construction et résolution |
| **5** | Résultats et visualisations |


---
## 1. Imports et paramètres globaux

In [ ]:
import os, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyomo.environ import *

# ── Paramètres de l'horizon ───────────────────────────────────
T             = [1, 2, 3, 4]
T_max         = 4
t_juillet     = 2
t_aout        = [3]
noms_t        = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}

# ── Paramètres financiers ─────────────────────────────────────
TARGET        = 2_406_580   # 50 % de la valeur initiale réelle
C_DISP        = 0.15        # coût de déstockage = 15 % de la valeur
                            # (frais de retour fournisseur / mise au rebut nets)
PENALTY_LS    = 5.0         # pénalité vente perdue = 5× le prix unitaire

# ── Mapping semaines FY TE → périodes du modèle ───────────────
# FY TE commence en octobre.
# FY2026 W36-39 = Juin | W40-43 = Juillet | W44-47 = Août | W48-51 = Septembre
week_to_t = {}
for w in range(36, 40): week_to_t[f"2026/{w}"] = 1
for w in range(40, 44): week_to_t[f"2026/{w}"] = 2
for w in range(44, 48): week_to_t[f"2026/{w}"] = 3
for w in range(48, 52): week_to_t[f"2026/{w}"] = 4

print("✅  Paramètres globaux définis")
print(f"   TARGET        : {TARGET:>12,.0f} €  (−50 % du stock initial)")
print(f"   Coût DISP     : {C_DISP*100:.0f} % de la valeur unitaire")
print(f"   Horizon       : {list(noms_t.values())}")


---
## 2. Modèle MILP (ConcreteModel Pyomo)

### Formulation mathématique

**Indices :**
- $i \in MP$ : matières premières
- $j \in SF$ : semi-finis
- $l \in PF$ : produits finis
- $t \in \{1,\ldots,T\}$ : périodes (mois)

**Variables de décision :**

| Variable | Domaine | Signification |
|----------|---------|---------------|
| $S_{i,t}^{MP},\, S_{j,t}^{SF},\, S_{l,t}^{PF}$ | $\mathbb{R}^+$ | Stock en fin de période $t$ |
| $O_{i,t}^{MP}$ | $\mathbb{R}^+$ | Nouvelles commandes MP lancées en $t$ |
| $DISP_{i,t}^{MP}$ | $\mathbb{R}^+$ | Quantité MP déstockée (retour / rebut) en $t$ |
| $DISP_{j,t}^{SF}$ | $\mathbb{R}^+$ | Quantité SF déstockée en $t$ |
| $DISP_{l,t}^{PF}$ | $\mathbb{R}^+$ | Quantité PF déstockée en $t$ |
| $LS_{l,t}$ | $\mathbb{R}^+$ | Ventes perdues PF en $t$ (variable de faisabilité) |

**Fonction objectif :**

$$\min Z = \underbrace{\sum_t \sum_k p_k S_{k,t}}_{\text{coût immobilisation stock}} + \underbrace{c_{disp} \sum_t \sum_k p_k DISP_{k,t}}_{\text{coût de déstockage}} + \underbrace{P_{LS} \sum_t \sum_l p_l LS_{l,t}}_{\text{pénalité ventes perdues}}$$

**Contraintes :**

$$\text{C1 (bilan MP)} \quad S_{i,t}^{MP} = S_{i,t-1}^{MP} + OC_{i,t} + O_{i,\,t-LT_i}^{MP} - DISP_{i,t}^{MP}$$

$$\text{C2 (bilan SF)} \quad S_{j,t}^{SF} = S_{j,t-1}^{SF} + WIP_{j,t}^{SF} - DISP_{j,t}^{SF}$$

$$\text{C3 (bilan PF)} \quad S_{l,t}^{PF} = S_{l,t-1}^{PF} + WIP_{l,t}^{PF} - D_{l,t} + LS_{l,t} - DISP_{l,t}^{PF}$$

$$\text{C5 (cible fiscale)} \quad \sum_k p_k S_{k,T} \leq TARGET$$

$$\text{C6 (gel août)} \quad O_{i,t}^{MP} = 0 \quad \forall i,\, t \in t_{août}$$


In [ ]:
def construire_modele(I, J, L,
                      prix_MP, prix_SF, prix_PF,
                      stock_init_MP, stock_init_SF, stock_init_PF,
                      LT, OC, WIP_SF, WIP_PF, demande,
                      T, T_max, t_aout,
                      TARGET, C_DISP, PENALTY_LS):
    """
    Construit et retourne le modèle ConcreteModel Pyomo.
    Modèle d'optimisation des achats avec actions de déstockage.
    Pas de production, pas de BOM.
    """
    m = ConcreteModel("Optimisation_Achats_Destockage")

    # ── Ensembles ─────────────────────────────────────────────
    m.I   = Set(initialize=I)
    m.J   = Set(initialize=J)
    m.L   = Set(initialize=L)
    m.PER = Set(initialize=T)

    # ── Variables : stocks ────────────────────────────────────
    m.S_MP = Var(m.I, m.PER, domain=NonNegativeReals)
    m.S_SF = Var(m.J, m.PER, domain=NonNegativeReals)
    m.S_PF = Var(m.L, m.PER, domain=NonNegativeReals)

    # ── Variables : nouvelles commandes MP ────────────────────
    m.O_MP = Var(m.I, m.PER, domain=NonNegativeReals)

    # ── Variables : actions de déstockage (retour/rebut) ──────
    m.DISP_MP = Var(m.I, m.PER, domain=NonNegativeReals)
    m.DISP_SF = Var(m.J, m.PER, domain=NonNegativeReals)
    m.DISP_PF = Var(m.L, m.PER, domain=NonNegativeReals)

    # ── Variable de faisabilité : ventes perdues PF ───────────
    m.LS = Var(m.L, m.PER, domain=NonNegativeReals)

    # ── Fonction objectif ─────────────────────────────────────
    # Minimiser : immobilisation stock + coût déstockage + pénalité ventes perdues
    m.obj = Objective(
        expr=(
            sum(prix_MP[i] * m.S_MP[i,t] for i in I for t in T)
          + sum(prix_SF[j] * m.S_SF[j,t] for j in J for t in T)
          + sum(prix_PF[l] * m.S_PF[l,t] for l in L for t in T)
          + C_DISP * (
                sum(prix_MP[i] * m.DISP_MP[i,t] for i in I for t in T)
              + sum(prix_SF[j] * m.DISP_SF[j,t] for j in J for t in T)
              + sum(prix_PF[l] * m.DISP_PF[l,t] for l in L for t in T)
            )
          + PENALTY_LS * sum(prix_PF[l] * m.LS[l,t] for l in L for t in T)
        ),
        sense=minimize
    )

    # ── C1 : Bilan stock Matière Première ─────────────────────
    def c1(m, i, t):
        S_prev = stock_init_MP[i] if t == min(T) else m.S_MP[i, t-1]
        t_cmd  = t - min(LT[i], T_max)
        rcp    = m.O_MP[i, t_cmd] if t_cmd >= min(T) else 0
        return m.S_MP[i,t] == S_prev + OC[i][t] + rcp - m.DISP_MP[i,t]
    m.C1 = Constraint(m.I, m.PER, rule=c1)

    # ── C2 : Bilan stock Semi-Fini ────────────────────────────
    def c2(m, j, t):
        S_prev = stock_init_SF[j] if t == min(T) else m.S_SF[j, t-1]
        return m.S_SF[j,t] == S_prev + WIP_SF[j][t] - m.DISP_SF[j,t]
    m.C2 = Constraint(m.J, m.PER, rule=c2)

    # ── C3 : Bilan stock Produit Fini ─────────────────────────
    def c3(m, l, t):
        S_prev = stock_init_PF[l] if t == min(T) else m.S_PF[l, t-1]
        return m.S_PF[l,t] == S_prev + WIP_PF[l][t] - demande[(l,t)] + m.LS[l,t] - m.DISP_PF[l,t]
    m.C3 = Constraint(m.L, m.PER, rule=c3)

    # ── C5 : Cible de valeur de stock fin d'horizon ───────────
    m.C5 = Constraint(expr=(
        sum(prix_MP[i] * m.S_MP[i, T_max] for i in I)
      + sum(prix_SF[j] * m.S_SF[j, T_max] for j in J)
      + sum(prix_PF[l] * m.S_PF[l, T_max] for l in L)
      <= TARGET
    ))

    # ── C6 : Gel des commandes en août ────────────────────────
    for i in I:
        for t in t_aout:
            m.add_component(
                f"C6_freeze_{str(i).replace('-','_')}_{t}",
                Constraint(expr=m.O_MP[i,t] == 0)
            )

    return m

print("✅  Fonction construire_modele() définie")


---
## 3. Chargement des données

> Seul paramètre à modifier : `DOSSIER_DATA`

In [ ]:
DOSSIER_DATA = r"/content/sample_data"   # ◄ adapter si nécessaire
def f(nom): return os.path.join(DOSSIER_DATA, nom)

def norm(x): return str(x).strip().upper()

print(f"✅  Dossier : {os.path.abspath(DOSSIER_DATA)}")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.1 — Référentiel produits
# PN__STOCK__PRIX_EN_STOCK_TYPE_PN.XLSX
# Types SAP : ZROH=MP | ZHLB=SF | ZFRT=PF
# ══════════════════════════════════════════════════════
df_s = pd.read_excel(f("PN__STOCK__PRIX_EN_STOCK_TYPE_PN.XLSX"))
df_s.columns = df_s.columns.str.strip()
df_s = df_s[df_s['Material Type'].isin(['ZROH','ZHLB','ZFRT'])].copy()
df_s['Material'] = df_s['Material'].apply(norm)
df_s = df_s.drop_duplicates('Material').set_index('Material')

for col in ['Unrestricted Stock','Prix_Unitaire','Inv Value in EUR']:
    df_s[col] = pd.to_numeric(df_s[col], errors='coerce').fillna(0)

I = df_s[df_s['Material Type']=='ZROH'].index.tolist()
J = df_s[df_s['Material Type']=='ZHLB'].index.tolist()
L = df_s[df_s['Material Type']=='ZFRT'].index.tolist()

prix_MP       = df_s.loc[I, 'Prix_Unitaire'].to_dict()
stock_init_MP = df_s.loc[I, 'Unrestricted Stock'].to_dict()
prix_SF       = df_s.loc[J, 'Prix_Unitaire'].to_dict()
stock_init_SF = df_s.loc[J, 'Unrestricted Stock'].to_dict()
prix_PF       = df_s.loc[L, 'Prix_Unitaire'].to_dict()
stock_init_PF = df_s.loc[L, 'Unrestricted Stock'].to_dict()

val_MP  = df_s.loc[I,'Inv Value in EUR'].sum()
val_SF  = df_s.loc[J,'Inv Value in EUR'].sum()
val_PF  = df_s.loc[L,'Inv Value in EUR'].sum()
val_tot = val_MP + val_SF + val_PF

print("=" * 62)
print("  STOCK INITIAL")
print("=" * 62)
print(f"  MP  : {len(I):4d} références  →  {val_MP:>12,.0f}  €")
print(f"  SF  : {len(J):4d} références  →  {val_SF:>12,.0f}  €")
print(f"  PF  : {len(L):4d} références  →  {val_PF:>12,.0f}  €")
print(f"  {'─'*44}")
print(f"  TOTAL               →  {val_tot:>12,.0f}  €")
print(f"  Cible −50 %         →  {TARGET:>12,.0f}  €")
print(f"  À réduire           →  {val_tot-TARGET:>12,.0f}  €")
print("=" * 62)


In [ ]:
# ══════════════════════════════════════════════════════
# 3.2 — Lead Times (LEAD_TIME_FOURNISSEUR_AVEC_PN.xlsx)
# Converti : jours → mois, capé à T_max = 4
# ══════════════════════════════════════════════════════
df_lt = pd.read_excel(f("LEAD_TIME_FOURNISSEUR_AVEC_PN.xlsx"))
df_lt.columns = df_lt.columns.str.strip()
col_lt = next(c for c in df_lt.columns if 'LEAD' in c.upper())
df_lt['PN'] = df_lt['PN'].apply(norm)
lt_j = df_lt.groupby('PN')[col_lt].max().to_dict()

LT = {i: min(T_max, max(1, math.ceil(float(lt_j.get(i, 120)) / 30))) for i in I}

print(f"✅  Lead Times — {len(LT)} MP")
from collections import Counter
print(f"   Distribution (mois, cap {T_max}) :", dict(Counter(LT.values())))


In [ ]:
# ══════════════════════════════════════════════════════
# 3.3 — MOQ (PN_MLS.xls)
# ══════════════════════════════════════════════════════
try:
    df_moq = pd.read_excel(f("PN_MLS.xls"), engine='xlrd')
    df_moq.columns = df_moq.columns.str.strip()
    df_moq['Material'] = df_moq['Material'].apply(norm)
    moq_s = df_moq.drop_duplicates('Material').set_index('Material')['Cur. Minimum lot size']
    def get_moq(pn):
        v = moq_s.get(pn, None)
        return float(v) if (v is not None and not (isinstance(v,float) and math.isnan(v))) else 1.0
    MOQ = {i: get_moq(i) for i in I}
    print(f"✅  MOQ — {len(MOQ)} MP chargés")
except Exception as e:
    MOQ = {i: 1.0 for i in I}
    print(f"⚠️  MOQ : {e} → défaut = 1")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.4 — Demande client (DEMAND_ANALYSSIS.xlsx)
# Colonnes semaines FY TE : 2026/36-51
# Agrégation par somme → 4 mois
# ══════════════════════════════════════════════════════
df_dem = pd.read_excel(f("DEMAND_ANALYSSIS.xlsx"))
df_dem.columns = [str(c).strip() for c in df_dem.columns]
df_dem['MATERIAL'] = df_dem['MATERIAL'].apply(norm)
df_dem = df_dem[df_dem['MATERIAL'].isin(L)].drop_duplicates('MATERIAL').set_index('MATERIAL')

weeks_ok = [w for w in week_to_t if w in df_dem.columns]
for w in weeks_ok:
    df_dem[w] = pd.to_numeric(df_dem[w], errors='coerce').fillna(0)

demande = {}
for l in L:
    for t in T:
        cols_t = [w for w in weeks_ok if week_to_t[w] == t]
        if l in df_dem.index and cols_t:
            demande[(l, t)] = max(0, float(df_dem.loc[l, cols_t].sum()))
        else:
            demande[(l, t)] = 0.0

n_pf_dem = sum(1 for l in L if sum(demande[(l,t)] for t in T) > 0)
tot_dem  = sum(demande.values())
val_dem  = sum(prix_PF[l]*demande[(l,t)] for l in L for t in T)

print(f"✅  Demande — {len(weeks_ok)} semaines → 4 mois")
print(f"   PF avec demande > 0 : {n_pf_dem} / {len(L)}")
print(f"   Demande totale      : {tot_dem:>12,.0f}  unités")
print(f"   Valeur à satisfaire : {val_dem:>12,.0f}  €")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.5 — Pipeline SAP (WIP_Achats_En_Cours.xlsx)
# OC[i][t]     : achats MP déjà commandés, arrivant en t
# WIP_SF[j][t] : encours SF déjà lancés, terminés en t
# WIP_PF[l][t] : encours PF déjà lancés, terminés en t
# ══════════════════════════════════════════════════════
OC     = {i: {t: 0.0 for t in T} for i in I}
WIP_SF = {j: {t: 0.0 for t in T} for j in J}
WIP_PF = {l: {t: 0.0 for t in T} for l in L}

def charger(sheet, col_pn, col_qte, dico, valide):
    try:
        df = pd.read_excel(f("WIP_Achats_En_Cours.xlsx"), sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        df[col_pn] = df[col_pn].apply(norm)
        for _, row in df.iterrows():
            pn = row[col_pn]
            t  = int(row.get('Mois Modèle (t)', 0) or 0)
            q  = float(row.get(col_qte, 0) or 0)
            if pn in dico and t in T:
                dico[pn][t] += q
        print(f"   ✅  {sheet:<30} : {len(df):4d} lignes")
    except Exception as e:
        print(f"   ⚠️  {sheet:<30} : {e}")

print("Pipeline SAP :")
charger("Achats_En_Cours_MP",  "PN",    "Qté Commandée", OC,     I)
charger("WIP_Semi_Finis",      "PN SF", "Qté En-Cours",  WIP_SF, J)
charger("WIP_Produits_Finis",  "PN PF", "Qté En-Cours",  WIP_PF, L)

val_oc  = sum(prix_MP[i]*OC[i][t]     for i in I for t in T)
val_wsf = sum(prix_SF[j]*WIP_SF[j][t] for j in J for t in T)
val_wpf = sum(prix_PF[l]*WIP_PF[l][t] for l in L for t in T)
print(f"\n   Valeur OC  (achats en transit)   : {val_oc:>10,.0f} €")
print(f"   Valeur WIP SF (encours prod SF)   : {val_wsf:>10,.0f} €")
print(f"   Valeur WIP PF (encours prod PF)   : {val_wpf:>10,.0f} €")
print(f"   Total pipeline                     : {val_oc+val_wsf+val_wpf:>10,.0f} €")


---
## 4. Construction de l'instance et résolution

In [ ]:
# ══════════════════════════════════════════════════════
# Construction du modèle ConcreteModel
# ══════════════════════════════════════════════════════
print("Construction du modèle...")
m = construire_modele(
    I=I, J=J, L=L,
    prix_MP=prix_MP, prix_SF=prix_SF, prix_PF=prix_PF,
    stock_init_MP=stock_init_MP,
    stock_init_SF=stock_init_SF,
    stock_init_PF=stock_init_PF,
    LT=LT, OC=OC, WIP_SF=WIP_SF, WIP_PF=WIP_PF,
    demande=demande,
    T=T, T_max=T_max, t_aout=t_aout,
    TARGET=TARGET, C_DISP=C_DISP, PENALTY_LS=PENALTY_LS
)

n_var = sum(1 for v in m.component_objects(Var, active=True)
            for _ in v)
n_cst = sum(1 for c in m.component_objects(Constraint, active=True)
            for _ in c)

print(f"✅  Modèle construit")
print(f"   Variables   : {n_var:,.0f}")
print(f"   Contraintes : {n_cst:,.0f}")
print(f"   Type        : LP (pas de variables binaires → résolution rapide)")


In [ ]:
# ══════════════════════════════════════════════════════
# Résolution avec CBC (open-source)
# ══════════════════════════════════════════════════════
print("Résolution en cours...")
solver  = SolverFactory('cbc')
resultat = solver.solve(m, tee=False,
                        options={'seconds': 300, 'ratioGap': 0.001})
status  = str(resultat.solver.termination_condition)

print()
print("=" * 62)
print("  RÉSULTAT DE LA RÉSOLUTION")
print("=" * 62)
print(f"  Statut          : {status}")
if status in ('optimal','feasible'):
    print(f"  Valeur obj Z*   : {value(m.obj):>14,.0f}")
print("=" * 62)


---
## 5. Résultats et visualisations

In [ ]:
# ── Valeur de stock par période et par étage ──────────────────
rows = []
for t in T:
    vmp = sum(value(m.S_MP[i,t])*prix_MP[i] for i in I)
    vsf = sum(value(m.S_SF[j,t])*prix_SF[j] for j in J)
    vpf = sum(value(m.S_PF[l,t])*prix_PF[l] for l in L)
    dmp = sum(value(m.DISP_MP[i,t])*prix_MP[i] for i in I)
    dsf = sum(value(m.DISP_SF[j,t])*prix_SF[j] for j in J)
    dpf = sum(value(m.DISP_PF[l,t])*prix_PF[l] for l in L)
    ls  = sum(value(m.LS[l,t])*prix_PF[l] for l in L)
    rows.append({
        "Période": noms_t[t],
        "MP (€)": vmp, "SF (€)": vsf, "PF (€)": vpf,
        "Total stock (€)": vmp+vsf+vpf,
        "Déstockage (€)": dmp+dsf+dpf,
        "Ventes perdues (€)": ls
    })
df_res = pd.DataFrame(rows).set_index("Période")

# Valeur de départ (t=0)
val_t0 = val_tot

print("Trajectoire de valeur de stock par étage :")
display(df_res.applymap(lambda x: f"{x:,.0f}"))


In [ ]:
# ── Vérification de la cible C5 ───────────────────────────────
val_fin = df_res.loc[noms_t[T_max],"Total stock (€)"]
print(f"  Valeur stock fin {noms_t[T_max]:<9} : {val_fin:>14,.0f}  €")
print(f"  Cible TARGET                  : {TARGET:>14,.0f}  €")
print(f"  Stock initial                 : {val_tot:>14,.0f}  €")
print(f"  Réduction obtenue             : {val_tot-val_fin:>14,.0f}  €  ({(val_tot-val_fin)/val_tot*100:.1f}%)")
print()
if val_fin <= TARGET * 1.005:
    print("  ✅  Cible atteinte !")
else:
    print(f"  ❌  Écart résiduel : {val_fin - TARGET:>10,.0f} €")


In [ ]:
# ══════════════════════════════════════════════════════
# GRAPHIQUE 1 : Trajectoire de la valeur de stock
# ══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 5))

periodes = list(df_res.index)
for col, color, lw in [
    ("MP (€)","steelblue",1.5),
    ("SF (€)","darkorange",1.5),
    ("PF (€)","seagreen",1.5),
    ("Total stock (€)","black",2.5),
]:
    vals = [val_tot if col=="Total stock (€)" else
            (val_MP if col=="MP (€)" else (val_SF if col=="SF (€)" else val_PF))] +            df_res[col].tolist() if False else df_res[col].tolist()
    label = col.replace(" (€)","")
    ax.plot(periodes, df_res[col], marker="o", lw=lw, color=color, label=label)

ax.axhline(val_tot,  color="grey",  ls=":",  lw=1.5, label=f"Stock initial ({val_tot:,.0f} €)")
ax.axhline(TARGET,   color="red",   ls="--", lw=2.0, label=f"Cible −50 % ({TARGET:,.0f} €)")

ax.set_title("Trajectoire de la valeur de stock — Juin à Septembre",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Période"); ax.set_ylabel("Valeur du stock (€)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x/1e6:.1f} M€"))
ax.legend(loc="upper right"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════
# GRAPHIQUE 2 : Répartition du déstockage par étage
# ══════════════════════════════════════════════════════
tot_disp_MP = sum(value(m.DISP_MP[i,t])*prix_MP[i] for i in I for t in T)
tot_disp_SF = sum(value(m.DISP_SF[j,t])*prix_SF[j] for j in J for t in T)
tot_disp_PF = sum(value(m.DISP_PF[l,t])*prix_PF[l] for l in L for t in T)
tot_disp    = tot_disp_MP + tot_disp_SF + tot_disp_PF

print(f"Actions de déstockage recommandées par le modèle :")
print(f"   MP  : {tot_disp_MP:>12,.0f} €  ({tot_disp_MP/max(tot_disp,1)*100:.1f}%)")
print(f"   SF  : {tot_disp_SF:>12,.0f} €  ({tot_disp_SF/max(tot_disp,1)*100:.1f}%)")
print(f"   PF  : {tot_disp_PF:>12,.0f} €  ({tot_disp_PF/max(tot_disp,1)*100:.1f}%)")
print(f"   {'─'*38}")
print(f"   TOTAL : {tot_disp:>10,.0f} €")
print(f"   Coût de déstockage (×{C_DISP:.0%}) : {tot_disp*C_DISP:>10,.0f} €")

if tot_disp > 0:
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    etages = ["MP", "SF", "PF"]
    vals   = [tot_disp_MP, tot_disp_SF, tot_disp_PF]
    colors = ["steelblue","darkorange","seagreen"]
    bars   = ax2.barh(etages, vals, color=colors, edgecolor="white", height=0.5)
    ax2.bar_label(bars, fmt=lambda x: f"{x/1e3:,.0f} k€", padding=5)
    ax2.set_title("Valeur des actions de déstockage recommandées par étage", fontweight="bold")
    ax2.set_xlabel("Valeur (€)")
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x/1e3:.0f}k"))
    ax2.grid(axis="x", alpha=0.25); plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════
# TOP 20 — PNs à déstocquer en priorité (valeur)
# ══════════════════════════════════════════════════════
lignes = []

# MP
for i in I:
    d = sum(value(m.DISP_MP[i,t])*prix_MP[i] for t in T)
    if d > 1:
        lignes.append({"PN":i,"Étage":"MP","Prix (€)":prix_MP[i],
                        "Qté à déstocquer":sum(value(m.DISP_MP[i,t]) for t in T),
                        "Valeur déstockage (€)":d})
# SF
for j in J:
    d = sum(value(m.DISP_SF[j,t])*prix_SF[j] for t in T)
    if d > 1:
        lignes.append({"PN":j,"Étage":"SF","Prix (€)":prix_SF[j],
                        "Qté à déstocquer":sum(value(m.DISP_SF[j,t]) for t in T),
                        "Valeur déstockage (€)":d})
# PF
for l in L:
    d = sum(value(m.DISP_PF[l,t])*prix_PF[l] for t in T)
    if d > 1:
        lignes.append({"PN":l,"Étage":"PF","Prix (€)":prix_PF[l],
                        "Qté à déstocquer":sum(value(m.DISP_PF[l,t]) for t in T),
                        "Valeur déstockage (€)":d})

df_top = (pd.DataFrame(lignes)
            .sort_values("Valeur déstockage (€)", ascending=False)
            .head(20)
            .set_index("PN"))

print("Top 20 références — actions de déstockage prioritaires :")
fmt = {"Prix (€)":"{:,.2f}","Qté à déstocquer":"{:,.0f}","Valeur déstockage (€)":"{:,.0f}"}
display(df_top.style.format(fmt))


In [ ]:
# ══════════════════════════════════════════════════════
# RÉSUMÉ FINAL
# ══════════════════════════════════════════════════════
val_fin = df_res.loc[noms_t[T_max],"Total stock (€)"]
tot_ls  = sum(value(m.LS[l,t])*prix_PF[l] for l in L for t in T)

print()
print("=" * 65)
print("  RÉSUMÉ FINAL DE L'OPTIMISATION")
print("=" * 65)
print(f"  Stock initial (t=0)        : {val_tot:>14,.0f}  €")
print(f"  Stock optimal fin septembre : {val_fin:>14,.0f}  €")
print(f"  Réduction obtenue           : {val_tot-val_fin:>14,.0f}  €  ({(val_tot-val_fin)/val_tot*100:.1f}%)")
print(f"  Cible −50 %                 : {TARGET:>14,.0f}  €")
print(f"  Cible respectée             : {'✅  Oui' if val_fin<=TARGET*1.005 else '❌  Non'}")
print()
print(f"  Actions de déstockage totales : {tot_disp:>12,.0f}  €")
print(f"  Coût de déstockage (×{C_DISP:.0%})   : {tot_disp*C_DISP:>12,.0f}  €")
print(f"  Ventes perdues (valeur)       : {tot_ls:>12,.0f}  €")
print()
print(f"  Nouvelles commandes MP passées : {sum(value(m.O_MP[i,t]) for i in I for t in T):>8,.0f}  unités")
print("=" * 65)
